# Control variates for the pseudo-Cl covariance: minimal example

Two simulations are built from the same white pixel noise (`python/control_variate/cv_sims.py`):

    real = map2alm[ W alm2map( sqrt(C_l) n_lm ) ]     colour, then mask (expensive)
    alt  = sqrt(C_l) map2alm( W n_p )                 mask, then colour (cheap)

`alt` has an analytic covariance $\Sigma_{\rm alt}$, so $\hat\Sigma = {\rm cov}(real) - c\,{\rm cov}(alt) + c\,\Sigma_{\rm alt}$
is unbiased for any $c$, and its scatter is smaller when real and alt are correlated. Toy patch, about a minute.

In [ ]:
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from pspy import pspy_utils, so_cov, so_map, so_window

sys.path.insert(0, "../python/control_variate")
from cv_covariance import SqrtPS, sigma_ll
from cv_sims import bandpowers, make_branches, sqrt_cl_matrix
from cv_spin2_kernels import spin2_kernels

lmax, bin_size = 88, 10
theory = "../../../old/Planck_pspy/data/cosmo2017_10K_acc3_lensedCls.dat"

# a rectangular patch with 1 degree pixels, and an apodized window on it
template = so_map.car_template(3, -60, 60, -30, 30, 60.0)
box = so_map.car_template(1, -60, 60, -30, 30, 60.0)
box.data[:] = 0
box.data[6:-6, 6:-6] = 1
win = so_window.create_apodization(box, "C1", 3.0, use_rmax=True)

sqrt_cl = sqrt_cl_matrix(theory, lmax)
draw, real, alt, A = make_branches(template, win, win, sqrt_cl, lmax)

binning = tempfile.NamedTemporaryFile(suffix=".dat", delete=False).name
pspy_utils.create_binning_file(bin_size=bin_size, n_bins=lmax // bin_size + 2, file_name=binning)
lb = pspy_utils.read_binning_file(binning, lmax)[2]

In [ ]:
# the same noise through both branches
spectra = ("TT", "EE")
N = 4000
rng = np.random.default_rng(1)
X = {s: [] for s in spectra}      # real
Y = {s: [] for s in spectra}      # alt
for _ in range(N):
    n_p = draw(rng)
    for s, b in bandpowers(real(n_p), lmax, binning, spectra).items():
        X[s].append(b)
    for s, b in bandpowers(alt(n_p), lmax, binning, spectra).items():
        Y[s].append(b)
X = {s: np.array(v) for s, v in X.items()}
Y = {s: np.array(v) for s, v in Y.items()}

# how well alt tracks real, per bin: the variance of a covariance element drops by 1 - r_b^4
r_b = {s: np.array([np.corrcoef(X[s][:, b], Y[s][:, b])[0, 1] for b in range(len(lb))]) for s in spectra}
for s in spectra:
    plt.plot(lb, r_b[s], "o-", label=s)
plt.xlabel(r"$\ell_b$"); plt.ylabel(r"$r_b$ (real, alt)"); plt.ylim(0, 1.02); plt.legend();

In [ ]:
# analytic covariance of alt, checked against its own Monte Carlo covariance
sqrt_ps = SqrtPS.from_single_array(sqrt_cl, lmax=lmax)
coupling = spin2_kernels(win, win, lmax)
sigma_alt = {s: so_cov.bin_mat(sigma_ll(sqrt_ps, coupling, s, s, A_norm=A), binning, lmax) for s in spectra}

for s in spectra:
    plt.plot(lb, np.diag(np.cov(Y[s], rowvar=False)) / np.diag(sigma_alt[s]), "o-", label=s)
plt.axhline(1, color="k", lw=0.5)
plt.xlabel(r"$\ell_b$"); plt.ylabel("MC / analytic variance of alt"); plt.legend();

In [ ]:
# the estimator, with c = 1, on 40 independent batches of 100 pairs: plain Monte Carlo against
# the control variate. The scatter of a covariance element across batches is what improves.
n_batch = 40
n = N // n_batch
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, s in zip(axes, spectra):
    plain, cv = [], []
    for i in range(n_batch):
        x, y = X[s][i * n:(i + 1) * n], Y[s][i * n:(i + 1) * n]
        plain.append(np.diag(np.cov(x, rowvar=False)))
        cv.append(np.diag(np.cov(x, rowvar=False) - np.cov(y, rowvar=False) + sigma_alt[s]))
    gain = np.var(plain, axis=0, ddof=1) / np.var(cv, axis=0, ddof=1)
    ax.plot(lb, gain, "o-", label="measured")
    ax.plot(lb, 1 / (1 - r_b[s] ** 4), "k--", label=r"$1/(1-r_b^4)$")
    ax.set_yscale("log"); ax.set_title(s); ax.set_xlabel(r"$\ell_b$"); ax.legend()
axes[0].set_ylabel("variance reduction")
plt.tight_layout()

In the lowest bin $c=1$ is not optimal (the reduction is below 1): fit $c$ on a held-out set there, or use $c=0$.

For survey data use the survey windows in `make_branches`, and pass the cross-array $\sqrt{C_\ell}$ and the array lists to `SqrtPS` and `sigma_ll`. The estimator and the choice of $c$ are in the pISO pipeline scripts.